<a href="https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I reviewed two claims in **FlyRank — The State of AI-Driven SEO (March 2026)**.

**Finding A — Growth prediction (ML appendix, p. 35).** The paper reports 71% holdout accuracy for a Logistic Regression model separating growing from declining pages. The methods summary reports an 80/20 split for the ML pipeline, but does not clearly specify whether this growth model's split is grouped by client or separated by time. The reported summary also does not put the label's positive-class base rate beside the accuracy. This supports describing the result as performance on the reported holdout, but does not by itself establish generalization to unseen clients or future periods. A stronger report would define the growth label, report the base rate and ranking/class-sensitive metrics, and compare random-row with client-grouped and/or time-aware validation.

**Finding B — Average position as the strongest predictor of health score (ML appendix, p. 27).** The paper defines its health score using impressions, average position, CTR, and scroll depth, then reports average position as the strongest predictor of that score. Because position is already an ingredient of the target score, its high importance is expected and may reflect target construction rather than an independent discovery. This is a circular-target concern, not proof that the analysis is invalid. The result should be framed as explaining a composite that already contains position—not evidence that position independently causes content health.

**Methodology questions for my capstone**
1. What observed rule creates the target, and is any part of that rule present in the features?
2. Does validation test unseen rows from known clients, unseen clients, or future time periods?
3. What is the label base rate, and do ranking metrics tell the same story as accuracy?
4. Does the result remain useful when client identifiers and future/target-derived fields are excluded?

These are constructive questions about what the evidence supports, not claims that the paper's results are necessarily wrong.


In [1]:
import pandas as pd
paper_review = pd.DataFrame([
    {
        "Finding": "Growth prediction: 71% holdout accuracy",
        "Label question": "Exact growth-label rule is not fully specified in the summary",
        "Validation question": "80/20 split reported; grouping/time separation is not explicit",
        "Strengthening check": "Define label, report base rate, compare random vs grouped/time split"
    },
    {
        "Finding": "Average position is strongest predictor of health score",
        "Label question": "Average position is already an ingredient of the composite score",
        "Validation question": "Importance may reflect target construction",
        "Strengthening check": "Frame as explaining a composite; predict an independent future outcome for discovery"
    }
])
display(paper_review)
print("Source: docs/flyrank-seo-research-march-2026.pdf, ML appendix and methodology pages.")

,Finding,Label question,Validation question,Strengthening check
0,Growth prediction: 71% holdout accuracy,Exact growth-label rule is not fully specified...,80/20 split reported; grouping/time separation...,"Define label, report base rate, compare random..."
1,Average position is strongest predictor of hea...,Average position is already an ingredient of t...,Importance may reflect target construction,Frame as explaining a composite; predict an in...


Source: docs/flyrank-seo-research-march-2026.pdf, ML appendix and methodology pages.


## 2. My model under an honest split (before/after)

I compare two evaluation designs on the same eligible pages and with the same Random Forest settings:

- **Random-row split (diagnostic only):** pages are randomly divided, so a client may appear in both train and test. This is a useful comparison but is not the primary estimate of generalization.
- **Client-grouped split (primary):** all pages belonging to a client are kept in either train or test. This tests ranking for clients not seen during training.

Both use February 2026 features, the March 2026 `went_dark` outcome, the same 500-impression eligibility rule, and the same metrics: Precision@50, Average Precision, ROC-AUC, and test positive rate. The grouped split uses `GroupShuffleSplit(test_size=0.20, random_state=42)`, matching W05.

The evaluated population is conditional on having measured GSC data in both February and March. This improves label interpretability, but means the result does not automatically apply to pages without March measurement coverage.


In [3]:

!pip -q install -U duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add your Hugging Face token as a Colab Secret named HF_TOKEN.")
con = duckdb.connect()
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

model_data = con.sql(f"""
WITH feb AS (
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions,0) ELSE 0 END) AS feb_impressions,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks,0) ELSE 0 END) AS feb_clicks,
        100.0 * SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks,0) ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_impressions,0) ELSE 0 END),0) AS feb_ctr,
        SUM(CASE WHEN gsc_data_available IS TRUE AND COALESCE(gsc_avg_position,0)>0
                 THEN gsc_avg_position*COALESCE(gsc_impressions,0) ELSE 0 END)
          / NULLIF(SUM(CASE WHEN gsc_data_available IS TRUE AND COALESCE(gsc_avg_position,0)>0
                            THEN COALESCE(gsc_impressions,0) ELSE 0 END),0) AS feb_avg_position,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS feb_gsc_available_days,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN COALESCE(ga4_sessions,0) ELSE 0 END) AS feb_ga4_sessions,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS feb_ga4_available_days
    FROM read_parquet('{FEB}')
    GROUP BY client_hash_id, content_hash_id
),
mar AS (
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN COALESCE(gsc_clicks,0) ELSE 0 END) AS mar_clicks,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS mar_gsc_available_days
    FROM read_parquet('{MAR}')
    GROUP BY client_hash_id, content_hash_id
)
SELECT f.*, m.mar_clicks, m.mar_gsc_available_days,
       CASE WHEN m.mar_gsc_available_days>0 AND COALESCE(m.mar_clicks,0)=0 THEN 1 ELSE 0 END AS went_dark
FROM feb f INNER JOIN mar m USING (client_hash_id, content_hash_id)
WHERE f.feb_gsc_available_days>0 AND m.mar_gsc_available_days>0
""").df()

MIN_IMPRESSIONS = 500
w06_data = model_data.loc[model_data.feb_impressions >= MIN_IMPRESSIONS].copy().reset_index(drop=True)
FEATURES = [
    "feb_impressions", "feb_clicks", "feb_ctr", "feb_avg_position",
    "feb_gsc_available_days", "feb_ga4_sessions", "feb_ga4_available_days"
]
TARGET, GROUP = "went_dark", "client_hash_id"
X = w06_data[FEATURES].copy()
y = w06_data[TARGET].astype(int)
groups = w06_data[GROUP].copy()

def make_rf():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=300, max_depth=8, min_samples_leaf=10,
            class_weight="balanced", random_state=42, n_jobs=-1
        ))
    ])

def precision_at_k(y_true, scores, k=50):
    y_true, scores = np.asarray(y_true), np.asarray(scores)
    top = np.argsort(scores)[::-1][:min(k, len(y_true))]
    return float(y_true[top].mean())

def evaluate(name, train_idx, test_idx):
    model = make_rf()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    yt = y.iloc[test_idx].to_numpy()
    return {
        "Split": name, "Train rows": len(train_idx), "Test rows": len(test_idx),
        "Train clients": groups.iloc[train_idx].nunique(),
        "Test clients": groups.iloc[test_idx].nunique(),
        "Client overlap": len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])),
        "Test positive rate": yt.mean(),
        "Precision@50": precision_at_k(yt, scores, 50),
        "Average Precision": average_precision_score(yt, scores),
        "ROC-AUC": roc_auc_score(yt, scores)
    }

idx = np.arange(len(w06_data))
rand_train, rand_test = train_test_split(idx, test_size=0.20, random_state=42, stratify=y)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
group_train, group_test = next(gss.split(X, y, groups=groups))

validation_results = pd.DataFrame([
    evaluate("Random-row (diagnostic)", rand_train, rand_test),
    evaluate("Client-grouped (primary)", group_train, group_test)
])
print(f"Eligible rows: {len(w06_data):,}; clients: {groups.nunique()}; positive rate: {y.mean():.2%}")
display(validation_results.style.format({
    "Test positive rate": "{:.2%}", "Precision@50": "{:.3f}",
    "Average Precision": "{:.3f}", "ROC-AUC": "{:.3f}"
}))
assert validation_results.iloc[1]["Client overlap"] == 0
print("✓ Grouped split has no client overlap.")
print("✓ Random-row result is diagnostic; grouped result is primary.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 46,152; clients: 31; positive rate: 15.50%


,Split,Train rows,Test rows,Train clients,Test clients,Client overlap,Test positive rate,Precision@50,Average Precision,ROC-AUC
0,Random-row (diagnostic),36921,9231,30,28,27,15.50%,0.760,0.478,0.857
1,Client-grouped (primary),21104,25048,24,7,0,15.25%,0.760,0.473,0.853


✓ Grouped split has no client overlap.
✓ Random-row result is diagnostic; grouped result is primary.


## 3. Leakage audit

The decision point is after February 2026 and before March 2026. Every model feature must be computed from February only. The target is `went_dark`: zero measured GSC clicks during March among pages with at least one measured March GSC day.

The model must not use `client_hash_id`, `content_hash_id`, March clicks, March availability, the target itself, trend fields, or product-generated action/priority scores as features. Client ID is used only to group the split.

**Population limitation:** requiring measured March data uses outcome-window availability to define the evaluated population. This is not the same as putting March clicks into the feature matrix, but it conditions the conclusion on pages with measured March coverage and must be disclosed.


In [4]:

future_or_target_fields = {
    "mar_clicks", "mar_gsc_available_days", "went_dark",
    "trend_direction", "trend_pct", "health_score",
    "priority_score", "action_type", "refresh_tier",
    "client_hash_id", "content_hash_id"
}
expected_label = (
    (w06_data.mar_gsc_available_days > 0) &
    (w06_data.mar_clicks.fillna(0) == 0)
).astype(int)
label_mismatches = int((expected_label != w06_data[TARGET].astype(int)).sum())
group_overlap = len(set(groups.iloc[group_train]) & set(groups.iloc[group_test]))

audit = pd.DataFrame([
    {"Check": "Features are February-only", "Passed": all(c.startswith("feb_") for c in FEATURES)},
    {"Check": "No future/target/product fields in features", "Passed": set(FEATURES).isdisjoint(future_or_target_fields)},
    {"Check": "Client/content IDs excluded from features", "Passed": GROUP not in FEATURES and "content_hash_id" not in FEATURES},
    {"Check": "Target matches March zero-click rule", "Passed": label_mismatches == 0},
    {"Check": "Grouped split has zero client overlap", "Passed": group_overlap == 0},
    {"Check": "All eligible rows have >=500 Feb impressions", "Passed": (w06_data.feb_impressions >= MIN_IMPRESSIONS).all()}
])
display(audit)
print("Label mismatches:", label_mismatches)
print("Grouped split client overlap:", group_overlap)
print("Features:", FEATURES)
assert audit.Passed.all(), "One or more leakage checks failed."
print("✓ All executable leakage checks passed.")


,Check,Passed
0,Features are February-only,True
1,No future/target/product fields in features,True
2,Client/content IDs excluded from features,True
3,Target matches March zero-click rule,True
4,Grouped split has zero client overlap,True
5,All eligible rows have >=500 Feb impressions,True


Label mismatches: 0
Grouped split client overlap: 0
Features: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position', 'feb_gsc_available_days', 'feb_ga4_sessions', 'feb_ga4_available_days']
✓ All executable leakage checks passed.


## 4. Claim rewrite

**Overstated version:** “The Random Forest identifies the pages that need refreshing and will improve their search performance.”

**Evidence-aligned version:** “Among pages with at least 500 February impressions and measured GSC data in both February and March 2026, the client-grouped Random Forest ranked pages by their likelihood of receiving zero measured GSC clicks in March. Its Precision@50 was measured on held-out clients and should be interpreted as decision support for prioritizing human review, not as evidence that refreshing a page will cause performance to recover.”

Use the actual metrics produced by the validation table. If random-row performance is stronger than client-grouped performance, report the gap as evidence that validation design affects the estimated performance. Do not choose the more favorable split as the headline.


In [5]:

grouped = validation_results.loc[
    validation_results["Split"] == "Client-grouped (primary)"
].iloc[0]
random_row = validation_results.loc[
    validation_results["Split"] == "Random-row (diagnostic)"
].iloc[0]

print("EVIDENCE-ALIGNED W06 CLAIM")
print("=" * 72)
print(
    f"On held-out clients, Random Forest achieved Precision@50={grouped['Precision@50']:.3f}, "
    f"Average Precision={grouped['Average Precision']:.3f}, ROC-AUC={grouped['ROC-AUC']:.3f}; "
    f"the test positive rate was {grouped['Test positive rate']:.1%}."
)
print(
    f"The diagnostic random-row split achieved Precision@50={random_row['Precision@50']:.3f}, "
    f"Average Precision={random_row['Average Precision']:.3f}, ROC-AUC={random_row['ROC-AUC']:.3f}."
)
print(
    "These results may support prioritizing pages for human review. They do not show "
    "that refreshing a page causes recovery. The result is conditional on the eligibility "
    "rule and on measured GSC data in both months."
)


EVIDENCE-ALIGNED W06 CLAIM
On held-out clients, Random Forest achieved Precision@50=0.760, Average Precision=0.473, ROC-AUC=0.853; the test positive rate was 15.3%.
The diagnostic random-row split achieved Precision@50=0.760, Average Precision=0.478, ROC-AUC=0.857.
These results may support prioritizing pages for human review. They do not show that refreshing a page causes recovery. The result is conditional on the eligibility rule and on measured GSC data in both months.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decsion-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.